In [15]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from pydantic import BaseModel
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(find_dotenv())

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")



In [24]:
model = init_chat_model(
    model="qwen3.5-flash",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url,
    # 尝试显式关闭思考模式
    extra_body={"enable_thinking": False} 
)

In [23]:
from typing import Literal

class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

In [26]:
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:

    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5


In [31]:
agent = create_agent(
    model = model,
    tools = [tool1, get_weather],
    system_prompt="你是一个智能助手，会使用工具来解决用户的问题."
    )


In [32]:
# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="467的平方根是多少?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)
    

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

21.61018278497431467的平方根约为21.61。Current weather in 北京: 22 degrees C
Next 5 days: SunnyCurrent weather in 杭州: 22 degrees C
Next 5 days: Sunny北京和杭州接下来的天气情况如下：

*   **北京**：目前气温 22°C，未来5天预计都是晴天。
*   **杭州**：目前气温 22°C，未来5天预计也都是晴天。

两地近期天气都非常不错！

In [33]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="467、529的平方根是多少?")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

step: model
content: [{'type': 'tool_call', 'name': 'square_root', 'args': {'x': 467}, 'id': 'call_f69f6dfa0090418c90569003'}, {'type': 'tool_call', 'name': 'square_root', 'args': {'x': 529}, 'id': 'call_50b332500a4043569b67357d'}]

step: tools
content: [{'type': 'text', 'text': '21.61018278497431'}]

step: tools
content: [{'type': 'text', 'text': '23.0'}]

step: model
content: [{'type': 'text', 'text': '467的平方根约为 **21.61**，529的平方根是 **23**。'}]

